# Correlation analysis of the predictor set

### 1. Import data

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Repo root: cwd may be repo root, notebooks/, or notebooks/shap/
REPO_ROOT = Path.cwd()
for _ in range(5):
    if (REPO_ROOT / 'utils' / 'base_utils.py').exists():
        break
    REPO_ROOT = REPO_ROOT.parent
else:
    raise RuntimeError('Could not find repo root (utils/base_utils.py).')
sys.path.insert(0, str(REPO_ROOT))

import utils.base_utils as bu
from utils.macro_grouping import add_group_level, build_full_group_mapping

start_date = '1971-08-31'
end_date = '2025-06-30'

yields = bu.get_yields(type='kr', start=start_date, end=end_date,
                       maturities=[str(i) for i in range(12, 121) if i % 12 == 0])
forward = bu.get_forward_rates(yields)
xr = bu.get_excess_returns(yields, horizon=12).dropna()

fred_md_start_date = pd.to_datetime(start_date) - pd.DateOffset(months=6)
fred_md_raw = bu.get_fred_data(str(REPO_ROOT / 'data' / '2026-01-MD.csv'),
                               start=fred_md_start_date, end=end_date)

fred_md = fred_md_raw.shift(1)
fred_md = fred_md.drop(columns=['TWEXAFEGSMTHx', 'ACOGNO'])
fred_md = fred_md[start_date:end_date]

yields = yields.loc[yields.index <= xr.index[-1]]
forward = forward.loc[forward.index <= xr.index[-1]]
xr = xr.loc[xr.index <= xr.index[-1]]
fred_md = fred_md.loc[fred_md.index <= xr.index[-1]]

# Yields are still loaded above for XR and curve math; predictor matrix uses forwards only.
s2g = build_full_group_mapping(fred_md, forward, yields)
X = pd.concat([fred_md, forward], axis=1, keys=['fred', 'forward'])
col_names = X.columns.get_level_values(-1)
s2g = {s: s2g[s] for s in col_names if s in s2g}
X = add_group_level(X, s2g, level_name='group')
X = X.sort_index(axis=1, level='group')

dates = xr.index
print('X shape:', X.shape, '| dates:', dates[0].date(), '->', dates[-1].date())

### 2. Group-level correlation heatmap

Each **block** is collapsed to one monthly series: **time z-score every series in the block**, then take the **equal-weight mean** inside the block. Blocks are **FRED-MD themes** plus **Forward Rates** (`X` excludes spot Treasury yields — those are kept only where needed for excess returns). The heatmap is **Kendall’s τ** (rank correlation) of those composites. Set `GROUP_BLOCKS` to subset blocks (`None` = all groups present in `X`).


In [ ]:
import seaborn as sns
from mpl_toolkits.axes_grid1 import make_axes_locatable

assert isinstance(X.columns, pd.MultiIndex) and ('group' in X.columns.names), (
    'X requires MultiIndex columns with a level named `group`.'
)

# None = every group in X (FRED themes + Forward Rates). Or pass an explicit subset:
GROUP_BLOCKS = None
# GROUP_BLOCKS = {'Output and Income', 'Labor Market', 'Housing',
#                 'Consumption, Orders, and Inventories', 'Money and Credit',
#                 'Interest and Exchange Rates', 'Prices', 'Stock Market',
#                 'Forward Rates'}

preferred_order = [
    'Forward Rates',
    'Output and Income',
    'Labor Market',
    'Housing',
    'Consumption, Orders, and Inventories',
    'Money and Credit',
    'Interest and Exchange Rates',
    'Prices',
    'Stock Market', 
]

all_blocks = sorted(set(X.columns.get_level_values('group')))
if GROUP_BLOCKS is not None:
    if not isinstance(GROUP_BLOCKS, set):
        GROUP_BLOCKS = set(GROUP_BLOCKS)
    blocks = [b for b in all_blocks if b in GROUP_BLOCKS]
else:
    blocks = list(all_blocks)


def block_series(sub: pd.DataFrame) -> pd.Series:
    if sub.shape[1] == 0:
        raise ValueError('empty block')
    mu = sub.mean(axis=0)
    sig = sub.std(axis=0, ddof=0).replace(0, np.nan)
    z = (sub - mu) / sig
    z = z.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return z.mean(axis=1)


G = {}
for b in blocks:
    sub = X.loc[:, X.columns.get_level_values('group') == b]
    G[b] = block_series(sub)

G = pd.DataFrame(G, index=X.index)
C = G.corr(method='kendall', min_periods=30)

order = [b for b in preferred_order if b in C.columns]
order.extend([b for b in C.columns if b not in order])
C = C.reindex(index=order, columns=order)

n = len(order)
fig_h = max(6, 1.1 * n)

fig, ax = plt.subplots(figsize=(10, fig_h))

# Create a colorbar axis with the same height as the heatmap axis
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="4%", pad=0.15)

sns.heatmap(
    C,
    cmap='RdBu_r',
    vmin=-1.0,
    vmax=1.0,
    center=0.0,
    square=True,
    linewidths=1.0,
    linecolor='white',
    annot=True,
    fmt='.2f',
    annot_kws={'fontsize': 11},
    cbar=True,
    cbar_ax=cax,
    cbar_kws={"label": "Kendall's τ"},
    ax=ax,
)


plt.tight_layout()
plt.show()

print('Blocks (rows/cols):', list(C.columns))

### 3. Within group correlation

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

assert isinstance(X.columns, pd.MultiIndex) and ('group' in X.columns.names), (
    'X requires MultiIndex columns with a level named `group`.'
)

preferred_order = [
    'Output and Income',
    'Labor Market',
    'Housing',
    'Consumption, Orders, and Inventories',
    'Money and Credit',
    'Interest and Exchange Rates',
    'Prices',
    'Stock Market',
    'Forward Rates',
]

GROUP_BLOCKS = None
# GROUP_BLOCKS = {'Output and Income', 'Labor Market', 'Forward Rates'}

all_blocks = list(pd.Index(X.columns.get_level_values('group')).unique())

if GROUP_BLOCKS is not None:
    GROUP_BLOCKS = set(GROUP_BLOCKS)
    blocks = [b for b in preferred_order if b in GROUP_BLOCKS and b in all_blocks]
    blocks += [b for b in all_blocks if b in GROUP_BLOCKS and b not in blocks]
else:
    blocks = [b for b in preferred_order if b in all_blocks]
    blocks += [b for b in all_blocks if b not in blocks]


def clean_label(col):
    """
    Makes readable labels from MultiIndex columns.
    Adjust this if your series names are stored in a specific level.
    """
    if isinstance(col, tuple):
        parts = [str(x) for x in col if str(x) not in ['', 'nan']]
        # Remove group name from label if present
        parts = [p for p in parts if p not in preferred_order]
        return parts[-1] if parts else str(col)
    return str(col)


def standardized_corr(sub: pd.DataFrame, min_periods: int = 30) -> pd.DataFrame:
    """
    Standardizes each series over time and computes within-group Kendall's τ.
    Standardization is not strictly necessary for rank correlation, but keeps preprocessing
    consistent with the block-level correlation code.
    """
    mu = sub.mean(axis=0)
    sig = sub.std(axis=0, ddof=0).replace(0, np.nan)
    z = (sub - mu) / sig
    z = z.replace([np.inf, -np.inf], np.nan)
    z = z.fillna(0.0)
    return z.corr(method='kendall', min_periods=min_periods)


for group in blocks:
    sub = X.loc[:, X.columns.get_level_values('group') == group].copy()


    Cg = standardized_corr(sub, min_periods=30)

    labels = [clean_label(c) for c in Cg.columns]
    Cg.index = labels
    Cg.columns = labels

    n = Cg.shape[0]
    fig_size = max(5, min(18, 0.45 * n))

    fig, ax = plt.subplots(figsize=(fig_size + 1.5, fig_size))

    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="4%", pad=0.15)

    sns.heatmap(
        Cg,
        cmap='RdBu_r',
        vmin=-1.0,
        vmax=1.0,
        center=0.0,
        square=True,
        linewidths=0.5,
        linecolor='white',
        annot=True,
        fmt='.2f',
        annot_kws={'fontsize': 8},
        cbar=True,
        cbar_ax=cax,
        cbar_kws={"label": "Kendall's τ"},
        ax=ax,
    )



    ax.tick_params(axis='x', labelrotation=45)
    ax.tick_params(axis='y', labelrotation=0)

    plt.tight_layout()
    plt.show()

    abs_corr = Cg.where(~np.eye(n, dtype=bool)).abs().stack()
    print(f"\n{group}")
    print(f"Number of variables: {n}")
    print(f"Mean |correlation|:   {abs_corr.mean():.3f}")
    print(f"Median |correlation|: {abs_corr.median():.3f}")
    print(f"Max |correlation|:    {abs_corr.max():.3f}")
    print(f"Share |corr| > 0.5:   {(abs_corr > 0.5).mean():.1%}")
    print(f"Share |corr| > 0.7:   {(abs_corr > 0.7).mean():.1%}")

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage

assert isinstance(X.columns, pd.MultiIndex) and ('group' in X.columns.names), (
    'X requires MultiIndex columns with a level named `group`.'
)

preferred_order = [
    'Output and Income',
    'Labor Market',
    'Housing',
    'Consumption, Orders, and Inventories',
    'Money and Credit',
    'Interest and Exchange Rates',
    'Prices',
    'Stock Market',
    'Forward Rates',
]

def block_series(sub: pd.DataFrame) -> pd.Series:
    mu = sub.mean(axis=0)
    sig = sub.std(axis=0, ddof=0).replace(0, np.nan)
    z = (sub - mu) / sig
    z = z.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return z.mean(axis=1)

blocks = [g for g in preferred_order if g in set(X.columns.get_level_values('group'))]

G = pd.DataFrame({
    g: block_series(X.loc[:, X.columns.get_level_values('group') == g])
    for g in blocks
}, index=X.index)

C_group = G.corr(method='kendall', min_periods=30)

D_group = 1 - C_group.abs()
np.fill_diagonal(D_group.values, 0.0)

link_group = linkage(squareform(D_group.values, checks=False), method='average')

sns.clustermap(
    C_group,
    row_linkage=link_group,
    col_linkage=link_group,
    cmap='RdBu_r',
    vmin=-1,
    vmax=1,
    center=0,
    annot=True,
    fmt='.2f',
    linewidths=1.0,
    linecolor='white',
    figsize=(8, 8),
    cbar_kws={"label": "Kendall's τ"}
)

plt.suptitle(
    'Hierarchical clustering of predictor groups',
    fontsize=14,
    fontweight='bold',
    y=1.02
)

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

assert isinstance(X.columns, pd.MultiIndex) and ('group' in X.columns.names), (
    'X requires MultiIndex columns with a level named `group`.'
)

# Coarser economic grouping
coarse_map = {
    # Yield curve
    'Forward Rates': 'Forward Rates',

    # Real activity
    'Output and Income': 'Real Activity',
    'Labor Market': 'Real Activity',
    'Housing': 'Real Activity',
    'Consumption, Orders, and Inventories': 'Real Activity',

    # Nominal / monetary / financial conditions
    'Prices': 'Prices',
    'Money and Credit': 'Money and Credit',
    'Interest and Exchange Rates': 'Interest and Exchange Rates',

    # Equity market
    'Stock Market': 'Stock Market',
}

coarse_order = [
    'Forward Rates',
    'Real Activity',
    'Prices',
    'Money and Credit',
    'Interest and Exchange Rates',
    'Stock Market',
]


def block_series(sub: pd.DataFrame) -> pd.Series:
    """
    Standardize each individual series over time, then take the equal-weighted
    mean within the block at each date.
    """
    if sub.shape[1] == 0:
        raise ValueError('empty block')

    mu = sub.mean(axis=0)
    sig = sub.std(axis=0, ddof=0).replace(0, np.nan)

    z = (sub - mu) / sig
    z = z.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    return z.mean(axis=1)


# Map each column to a coarse group
fine_groups = X.columns.get_level_values('group')
coarse_groups = pd.Index([coarse_map.get(g, np.nan) for g in fine_groups], name='coarse_group')

# Keep only columns that are mapped
keep = ~pd.isna(coarse_groups)
X_coarse = X.loc[:, keep].copy()
coarse_groups = coarse_groups[keep]

# Build one equal-weighted standardized composite per coarse group
G_coarse = {}

for cg in coarse_order:
    cols = coarse_groups == cg
    if cols.sum() > 0:
        G_coarse[cg] = block_series(X_coarse.loc[:, cols])

G_coarse = pd.DataFrame(G_coarse, index=X.index)

# Correlation between coarse blocks
C_coarse = G_coarse.corr(method='kendall', min_periods=30)
C_coarse = C_coarse.reindex(index=coarse_order, columns=coarse_order)

# Plot
fig, ax = plt.subplots(figsize=(7, 6))

divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="4%", pad=0.15)

sns.heatmap(
    C_coarse,
    cmap='RdBu_r',
    vmin=-1.0,
    vmax=1.0,
    center=0.0,
    square=True,
    linewidths=1.0,
    linecolor='white',
    annot=True,
    fmt='.2f',
    annot_kws={'fontsize': 11},
    cbar=True,
    cbar_ax=cax,
    cbar_kws={"label": "Kendall's τ"},
    ax=ax,
)


ax.tick_params(axis='x', labelrotation=45)
ax.tick_params(axis='y', labelrotation=0)

plt.tight_layout()
plt.show()

print(C_coarse)